# 기대 수명 데이터 분석

전 세계 기대 수명을 탐구하는 두 가지 데이터셋:
- **Gapminder** (1952-2007): country, year, population, continent, lifeExp, gdpPercap
- **WHO 기대 수명 데이터** (2000-2015): 193개국, 22개 지표 (사망률, BMI, GDP, 교육 연수 등)

이 워크북은 **Python**과 **R** 모두에서 CSV를 가져오고 분석하는 과정을 시연합니다.

## 1. 설정: 패키지 설치 및 데이터셋 다운로드

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('pandas + plotly 설치 완료')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"이미 존재함: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"다운로드 완료 {name}: {lines} 줄")

## 2. Gapminder: Python을 이용한 데이터 탐색

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"형상(Shape): {gap.shape}")
print(f"대륙 목록: {sorted(gap['continent'].unique())}")
print(f"연도 범위: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# 대륙별 시간에 따른 기대 수명 변화
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='대륙별 기대 수명 (1952-2007)',
              labels={'lifeExp': '기대 수명 (년)', 'year': '연도'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# 1인당 GDP 대비 기대 수명 (2007), 버블 크기 = 인구수
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='1인당 GDP 대비 기대 수명 (2007)',
                 labels={'gdpPercap': '1인당 GDP (로그 스케일)', 'lifeExp': '기대 수명'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: R을 이용한 데이터 탐색

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# 대륙별 기대 수명 분포 (상자 수염 그림)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "대륙별 기대 수명",
        xlab = "대륙", ylab = "기대 수명 (년)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# 기대 수명 증가폭 상위 10개국 (1952 vs 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "상위 10개국: 기대 수명 증가폭 (1952-2007)",
        xlab = "증가 연수",
        col = "#00CC96", border = NA)

## 4. WHO 기대 수명: Python을 이용한 데이터 탐색

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"형상(Shape): {who.shape}")
print(f"열 목록: {list(who.columns)}")
print(f"\n결측치 (상위 5개):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# 개발도상국 vs 선진국: 사전 구간화된 기대 수명 분포
# 명시적인 막대 좌표는 브라우저 Plotly 브리지를 통해 일관되게 렌더링됩니다.
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='기대 수명: 개발도상국 vs 선진국',
             labels={'Life expectancy': '기대 수명 (년)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# 교육 연수 대비 기대 수명
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='교육 연수 대비 기대 수명 (2014)',
                 labels={'Life expectancy': '기대 수명 (년)',
                         'Schooling': '교육 연수 (년)'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. WHO 기대 수명: R을 이용한 데이터 탐색

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\n국가 수:", length(unique(who$Country)))
cat("\n연도 범위:", range(who$Year))

In [ ]:
# 상관관계: 성인 사망률 대비 기대 수명
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "성인 사망률 대비 기대 수명",
     xlab = "성인 사망률 (1,000명당)",
     ylab = "기대 수명 (년)")
legend("topright", legend = c("선진국", "개발도상국"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# 단순 선형 모델: 기대 수명을 예측하는 요인은 무엇인가?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## 주요 발견 사항

- 기대 수명은 전 세계적으로 증가했으나 대륙 간 큰 격차는 여전히 존재합니다.
- GDP와 교육 연수는 기대 수명을 예측하는 강력한 양의 상관 요인입니다.
- 성인 사망률은 가장 강력한 음의 상관 요인입니다.
- 개발도상국은 결과(기대 수명)에서 훨씬 더 큰 편차를 보입니다.